<a href="https://colab.research.google.com/github/ImNotDanish05/2026_Pembelajaran-Mesin_TI-3A/blob/main/Week_8/FiturSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Unduh dataset menggunakan wget
!wget "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip" -O diabetes_dataset.zip

--2026-10-20 01:28:41--  https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘diabetes_dataset.zip’

diabetes_dataset.zi     [   <=>              ]   3.02M  6.58MB/s    in 0.5s    

2026-10-20 01:28:42 (6.58 MB/s) - ‘diabetes_dataset.zip’ saved [3170254]



In [2]:
# Ekstrak file zip yang telah diunduh
!unzip -o diabetes_dataset.zip -d diabetes_data

Archive:  diabetes_dataset.zip
  inflating: diabetes_data/diabetic_data.csv  
  inflating: diabetes_data/IDS_mapping.csv  


Directory Dataset: <br><br>/content/diabetes_data/diabetic_data.csv

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Load Dataset
df = pd.read_csv('/content/diabetes_data/diabetic_data.csv')

# Tampilkan informasi dasar data
print(f"Dimensi awal dataset: {df.shape}")
display(df.head())


Dimensi awal dataset: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
# 2. Preprocessing Data
# Mengganti karakter '?' dengan NaN
df = df.replace('?', np.nan)

# Mengurangi beban memori/komputasi dengan menghapus kolom dengan terlalu banyak missing values
# dan kolom identifikasi yang tidak relevan untuk prediksi
kolom_dihapus = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
df = df.drop(columns=[col for col in kolom_dihapus if col in df.columns])

# Mengisi missing values sederhana untuk fitur kategorikal & numerik sisanya
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Target variabel (klasifikasi): readmitted
# Kita akan memprediksi apakah pasien akan kembali dirawat ('readmitted')
target_col = 'readmitted'

# Label Encoding untuk kolom objek (kategorikal)
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

# Memisahkan Fitur (X) dan Target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Dimensi fitur setelah preprocessing: {X.shape}")


Dimensi fitur setelah preprocessing: (101766, 44)


In [5]:
# 3. Split Dataset & Standardisasi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Skala fitur karena KNN berbasis jarak (sangat sensitif terhadap skala fitur)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Pembagian dan standarisasi data selesai.")


Pembagian dan standarisasi data selesai.


In [6]:
# 4. Model Training (KNN)
# Menggunakan k=5 sebagai nilai default awal yang umum
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# 5. Evaluasi Model
y_pred = knn.predict(X_test_scaled)

print("=== Hasil Evaluasi KNN Classifier ===\n")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))


=== Hasil Evaluasi KNN Classifier ===

Accuracy Score: 0.4963

Classification Report:
              precision    recall  f1-score   support

           0       0.17      0.11      0.14      2272
           1       0.40      0.39      0.40      7109
           2       0.60      0.64      0.62     10973

    accuracy                           0.50     20354
   macro avg       0.39      0.38      0.38     20354
weighted avg       0.48      0.50      0.49     20354

